# Sampling-based planning: the whole run

Runs every number and picture on `sampling-planning.md`: the worlds, one seeded run of each planner, the checks, the 10-seed tables and the animations. About 5 minutes.

In [1]:
import math
from shapely.geometry import LineString, box
import run, world
from check_planners import check, check_grid_loader
from collision import bisection_order

## The worlds
The grid project's warehouse is read from its PNG map and written as `worlds/warehouse.json`. Each world's picture is redrawn when its file changes.

In [2]:
worlds = {}
for arg in ["small-warehouse", "../grid-pathfinding/maps/warehouse.png", "office"]:
    s = world.load_world(arg)
    worlds[s.key] = s
    x0, y0, x1, y1 = s.bounds
    print(f"{s.key:16} {s.name:16} {x1 - x0:g} m x {y1 - y0:g} m, {len(s.blocked)} blocked areas, robot radius {s.radius} m, "
          f"step {s.step} m, edge resolution {s.eps} m, test pad {s.pad * 1000:.1f} mm")
    print(f"{'':16} start {s.start}, goal {s.goal}, free area for the centre {s.free_area:.1f} m2, "
          f"shortest possible path {run.shortest_possible(s):.1f} m")

small-warehouse  Small warehouse  20 m x 12.5 m, 6 blocked areas, robot radius 0.25 m, step 1.0 m, edge resolution 0.05 m, test pad 2.5 mm
                 start (2.0, 1.5), goal (2.0, 11.0), free area for the centre 159.5 m2, shortest possible path 12.8 m


warehouse        Warehouse floor  40 m x 25 m, 11 blocked areas, robot radius 0.25 m, step 1.0 m, edge resolution 0.05 m, test pad 2.5 mm


                 start (2.0, 1.0), goal (30.0, 23.0), free area for the centre 508.5 m2, shortest possible path 43.0 m


office           Office           12 m x 8 m, 11 blocked areas, robot radius 0.25 m, step 0.6 m, edge resolution 0.05 m, test pad 2.5 mm
                 start (1.0, 1.0), goal (11.0, 7.0), free area for the centre 59.2 m2, shortest possible path 12.9 m


## The small warehouse: one run per planner (seed 2), checks, animations

In [3]:
space, results = run.main("small-warehouse")

Small warehouse: 20 m x 12.5 m, robot radius 0.25 m, free area for the robot's centre 159.5 m2, step 1.0 m, edge resolution 0.05 m
start (2.0, 1.5), goal (2.0, 11.0), straight line 9.5 m, shortest possible path 12.8 m



seed 2
planner       samples   checks  path (m)  time (s)
RRT               331    2,828      24.5     0.026
RRT*            1,500   31,625      13.8     0.584
RRT-Connect       210    1,950      44.2     0.018
PRM               772   53,744      37.2     0.335
checks passed: every path found is collision-free, starts at the start and ends at the goal; RRT* cost never rises; PRM's route is the shortest in its roadmap


The numbers quoted under each planner:

In [4]:
ev = results["RRT"]["events"]
print(f"RRT: {1 + sum(len(e['set']) for e in ev)} nodes, {sum(not e['ok'] for e in ev)} steps blocked, "
      f"{sum(e['sample'] == space.goal for e in ev)} samples were the goal")
ev = results["RRT*"]["events"]
first = next(k for k, e in enumerate(ev, 1) if e["cost"])
nodes, rewires = {space.start}, 0
for e in ev:
    for child, _ in e["set"]:
        rewires += child in nodes
        nodes.add(child)
drops = [(k, round(ev[k - 2]["cost"], 1), round(e["cost"], 1), len(e["set"]) - 1)
         for k, e in enumerate(ev, 1) if k > first and ev[k - 2]["cost"] - e["cost"] > 2]
print(f"RRT*: first path at sample {first}, {ev[first - 1]['cost']:.1f} m; {len(nodes)} nodes, {rewires} rewires; "
      f"neighbour radius {min(e['radius'] for e in ev):.2f} to {max(e['radius'] for e in ev):.2f} m; "
      f"big drops (sample, before, after, rewired): {drops}")
ev = results["RRT-Connect"]["events"]
print(f"RRT-Connect: {1 + sum(len(e['set']) for e in ev if e['tree'] == 0)} nodes from the start, "
      f"{1 + sum(len(e['set']) for e in ev if e['tree'] == 1)} from the goal")
graph = results["PRM"]["graph"]
in_gap = [p for p in graph if 5.6 < p[0] < 6.4 and 5.5 < p[1] < 7.5]
crossing_x = lambda a, b: a[0] + (6.5 - a[1]) / (b[1] - a[1]) * (b[0] - a[0])  # where edge a-b crosses y = 6.5
through = [(a, b) for a in graph for b in graph[a] if a < b and (a[1] < 6.5) != (b[1] < 6.5) and crossing_x(a, b) < 17]
print(f"PRM: {sum(e['phase'] == 'sample' and e['ok'] for e in results['PRM']['events'])} points kept, "
      f"{sum(e['phase'] == 'sample' and not e['ok'] for e in results['PRM']['events'])} thrown away, "
      f"{sum(len(e['edges']) for e in results['PRM']['events'] if e['phase'] == 'connect')} roadmap edges; "
      f"points in the gap: {len(in_gap)}; edges through the gap: {len(through)}")

RRT: 164 nodes, 169 steps blocked, 16 samples were the goal
RRT*: first path at sample 331, 22.5 m; 895 nodes, 819 rewires; neighbour radius 1.00 to 1.00 m; big drops (sample, before, after, rewired): [(630, 22.1, 14.5, 3)]
RRT-Connect: 44 nodes from the start, 51 from the goal
PRM: 500 points kept, 272 thrown away, 2881 roadmap edges; points in the gap: 1; edges through the gap: 0


Why test both ends first, then the middle: the tests each blocked RRT step needed, in that order and point by point from the near end.

In [5]:
steps = [(e["near"], e["new"]) for e in results["RRT"]["events"] if not e["ok"]]
ordered, in_line = [], []
for a, b in steps:
    before = space.checks
    assert not space.segment_free(a, b)
    ordered.append(space.checks - before)
    n = max(1, math.ceil(math.dist(a, b) / space.eps))
    in_line.append(next(i + 1 for i in range(n + 1) if not space.free((a[0] + i / n * (b[0] - a[0]), a[1] + i / n * (b[1] - a[1])))))
print(f"{len(steps)} blocked steps: {sum(ordered) / len(steps):.1f} tests each, ends first then halving; "
      f"{sum(in_line) / len(steps):.1f} tests each, point by point from the near end")

169 blocked steps: 1.1 tests each, ends first then halving; 8.0 tests each, point by point from the near end


Why the point test uses a robot 2.5 mm larger: the office run again with the true size.

In [6]:
office = world.load_world("office")
office.tested = office.cfree  # test points with the robot's true size
try:
    check(office, run.run_all(office))
    print("all paths pass without the pad")
except AssertionError as err:
    print("without the pad:", err)
    walls = office.free_region.boundary
    path = run.run_all(office)["RRT-Connect"]["path"]
    gap = min(LineString([a, b]).distance(walls) for a, b in zip(path, path[1:]))
    print(f"closest approach of that RRT-Connect path to a wall: {gap:.4f} m, with a robot radius of 0.25 m")

without the pad: RRT-Connect: segment (1.6693218035580575, 2.5846947902160506)-(1.776590417659611, 3.1750281235360758) hits an obstacle


closest approach of that RRT-Connect path to a wall: 0.2498 m, with a robot radius of 0.25 m


The way round the right end, if the 0.8 m gap were closed:

In [7]:
closed = dict(name="gap closed", bounds=[0, 0, 20, 12.5], robot_radius=0.25, start=space.start, goal=space.goal,
              free_region=space.free_region.difference(box(5.5, 5.5, 6.5, 7.5)))
print(f"shortest path with the gap closed: {run.shortest_possible(world.Space(closed)):.1f} m")

shortest path with the gap closed: 34.2 m


## Ten seeds
One run can be lucky. Medians over seeds 1 to 10, and how many runs found a path shorter than twice the shortest possible (here: through the gap).

In [8]:
print(run.seed_sweep(space))
print()
print(run.prm_sweep(space))

planner       samples   checks  path (m)  time (s)  under 2x best
RRT               478    3,591      42.2     0.027        3 of 10
RRT*            1,500   30,563      36.7     0.363        3 of 10
RRT-Connect       172    1,692      42.5     0.008        4 of 10
PRM               780   53,608      36.8     0.280        4 of 10



 samples  under 2x best  median route (m)  median checks
     250        4 of 10              36.5         38,139
     500        4 of 10              36.8         53,608
    1000       10 of 10              13.9         77,318
    2000        9 of 10              13.5        113,010


## The same code on the other two worlds

In [9]:
for key in ["warehouse", "office"]:
    run.main(key)
    print()

Warehouse floor: 40 m x 25 m, robot radius 0.25 m, free area for the robot's centre 508.5 m2, step 1.0 m, edge resolution 0.05 m
start (2.0, 1.0), goal (30.0, 23.0), straight line 35.6 m, shortest possible path 43.0 m



seed 2
planner       samples   checks  path (m)  time (s)
RRT             6,018   26,927      64.3     1.398
RRT*           12,036  232,382      52.6    11.041
RRT-Connect       550    4,168      46.7     0.030
PRM             3,013  165,324      44.9     1.128
checks passed: every path found is collision-free, starts at the start and ends at the goal; RRT* cost never rises; PRM's route is the shortest in its roadmap



Office: 12 m x 8 m, robot radius 0.25 m, free area for the robot's centre 59.2 m2, step 0.6 m, edge resolution 0.05 m
start (1.0, 1.0), goal (11.0, 7.0), straight line 11.7 m, shortest possible path 12.9 m



seed 2
planner       samples   checks  path (m)  time (s)
RRT               175      921      16.3     0.005
RRT*            1,500   17,093      13.5     0.271
RRT-Connect     1,081    2,137      17.0     0.019
PRM               825   35,055      13.8     0.180
checks passed: every path found is collision-free, starts at the start and ends at the goal; RRT* cost never rises; PRM's route is the shortest in its roadmap


## All checks

In [10]:
assert bisection_order(4) == [4, 0, 2, 1, 3]
check_grid_loader()
for key, s in worlds.items():
    assert not check(s, run.run_all(s)), f"{key}: a planner found no path"
    print(key, "ok")

small-warehouse ok


warehouse ok


office ok
